In [ ]:
using LinearAlgebra, Ferrite, ModularEIT

In [ ]:
n = 63 
grid = generate_grid(Quadrilateral, (n, n));
∂Ω = union(getfacetset.((grid,), ["left", "top", "right", "bottom"])...)
fe  = FerriteFESpace{RefQuadrilateral}(grid,2,3,∂Ω)

In [ ]:
assemble_boundary_matrices(fe)

In [ ]:
fe.BDO.MΓ

In [ ]:
function assemble_boundary_M_K(facetvalues::FacetValues, dh::DofHandler, ∂Ω,b_dofs)
    ndof = ndofs(dh)
    M = allocate_matrix(fe.dh)
    K = allocate_matrix(fe.dh)
    n_basefuncs = getnbasefunctions(facetvalues)
    Me = zeros(n_basefuncs, n_basefuncs)
    Ke = zeros(n_basefuncs, n_basefuncs)

    for facet in FacetIterator(dh, ∂Ω)
        fill!(Me, 0.0); fill!(Ke, 0.0)
        reinit!(facetvalues, facet)
        dofs = celldofs(facet)
        for q in 1:getnquadpoints(facetvalues)
            dΓ = getdetJdV(facetvalues, q)
            for i in 1:n_basefuncs
                φᵢ  = shape_value(facetvalues, q, i)
                ∇φᵢ = shape_gradient(facetvalues, q, i)  # tangential grad on the facet manifold
                for j in 1:n_basefuncs
                    φⱼ  = shape_value(facetvalues, q, j)
                    ∇φⱼ = shape_gradient(facetvalues, q, j)
                    Me[i, j] += φᵢ * φⱼ * dΓ
                    Ke[i, j] += (∇φᵢ ⋅ ∇φⱼ) * dΓ
                end
            end
        end
        for (a, A) in enumerate(dofs), (b, B) in enumerate(dofs)
            M[A, B] += Me[a, b]
            K[A, B] += Ke[a, b]
        end
    end
    return M[b_dofs, b_dofs], K[b_dofs, b_dofs]
end

function assemble_boundary_M_K(fe::FerriteFESpace)
    assemble_boundary_M_K(fe.facetvalues, fe.dh, fe.∂Ω, fe.b_dofs)
end

In [ ]:
b_dofs = fe.b_dofs
function get_i_dofs(b_dofs, n::Integer)
    is_boundary = falses(n)
    is_boundary[b_dofs] .= true
    return findall(!, is_boundary)
end
i_dofs = get_i_dofs(b_dofs, fe.n)

In [ ]:
K = fe.K
Kii = K[i_dofs, i_dofs]
kbb = K[b_dofs, b_dofs]
Kib = K[i_dofs, b_dofs]
Kbi = K[b_dofs, i_dofs]

In [ ]:
M_Γ, K_Γ = assemble_boundary_M_K(fe::FerriteFESpace)

In [ ]:
M_test = M_Γ |> Symmetric

In [ ]:
Mb, Kb = M_Γ |> Matrix |> Symmetric , K_Γ |> Matrix |> Symmetric

In [ ]:
λM, _ = eigen(Mb)
λK, _ = eigen(Kb)
println("$(minimum(λM))")
println("$(maximum(λK))")

In [ ]:
λ, Φ = eigen(Kb, Mb) 

In [ ]:
minimum(λ)

In [ ]:
A(s) = Mb * Φ * Diagonal((1.0 .+ λ).^s) * Φ' * Mb |> Symmetric

In [ ]:
Hn½ = A(-0.5)
H½ = A(0.5)

In [ ]:
H½inv = inv(H½)

In [ ]:
maximum(Mb * H½inv * Mb - Hn½)